# Advanced Pandas Operations — Week 2 Day 2

In [5]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('Downloads/Sample - Superstore.csv',encoding="cp1252")

# Inspect the dataset
print("Dataset shape:", df.shape)

display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Data types:
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


In [6]:
# Create a working copy
df = df.copy()

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Convert date columns
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

# Check missing values and duplicates
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

# Inspect numerical columns
display(df[["sales", "quantity", "discount", "profit"]].describe())

Missing values:
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub-category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

Duplicate rows: 0


,sales,quantity,discount,profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [7]:
# Create a sales DataFrame
sales_df = df[
    ["order_id", "customer_id", "region", "sales", "profit"]
].head(100).copy()

# Create a customer-region mapping DataFrame
customer_df = (
    df[["customer_id", "segment", "region"]]
    .drop_duplicates(subset="customer_id")
    .head(60)
    .copy()
)

print("Sales DataFrame:")
display(sales_df.head())

print("Customer DataFrame:")
display(customer_df.head())

Sales DataFrame:


,order_id,customer_id,region,sales,profit
0,CA-2016-152156,CG-12520,South,261.9600,41.9136
1,CA-2016-152156,CG-12520,South,731.9400,219.5820
2,CA-2016-138688,DV-13045,West,14.6200,6.8714
3,US-2015-108966,SO-20335,South,957.5775,-383.0310
4,US-2015-108966,SO-20335,South,22.3680,2.5164


Customer DataFrame:


,customer_id,segment,region
0,CG-12520,Consumer,South
2,DV-13045,Corporate,West
3,SO-20335,Consumer,South
5,BH-11710,Consumer,West
12,AA-10480,Consumer,South


In [8]:
inner_join = pd.merge(
    sales_df,
    customer_df,
    on="customer_id",
    how="inner",
    suffixes=("_sales", "_customer")
)

display(inner_join.head())

print("Inner join shape:", inner_join.shape)

,order_id,customer_id,region_sales,sales,profit,segment,region_customer
0,CA-2016-152156,CG-12520,South,261.9600,41.9136,Consumer,South
1,CA-2016-152156,CG-12520,South,731.9400,219.5820,Consumer,South
2,CA-2016-138688,DV-13045,West,14.6200,6.8714,Corporate,West
3,US-2015-108966,SO-20335,South,957.5775,-383.0310,Consumer,South
4,US-2015-108966,SO-20335,South,22.3680,2.5164,Consumer,South


Inner join shape: (100, 7)


In [9]:
left_join = pd.merge(
    sales_df,
    customer_df,
    on="customer_id",
    how="left",
    suffixes=("_sales", "_customer")
)

display(left_join.head())

print("Left join shape:", left_join.shape)

,order_id,customer_id,region_sales,sales,profit,segment,region_customer
0,CA-2016-152156,CG-12520,South,261.9600,41.9136,Consumer,South
1,CA-2016-152156,CG-12520,South,731.9400,219.5820,Consumer,South
2,CA-2016-138688,DV-13045,West,14.6200,6.8714,Corporate,West
3,US-2015-108966,SO-20335,South,957.5775,-383.0310,Consumer,South
4,US-2015-108966,SO-20335,South,22.3680,2.5164,Consumer,South


Left join shape: (100, 7)


In [10]:
right_join = pd.merge(
    sales_df,
    customer_df,
    on="customer_id",
    how="right",
    suffixes=("_sales", "_customer")
)

display(right_join.head())

print("Right join shape:", right_join.shape)

,order_id,customer_id,region_sales,sales,profit,segment,region_customer
0,CA-2016-152156,CG-12520,South,261.9600,41.9136,Consumer,South
1,CA-2016-152156,CG-12520,South,731.9400,219.5820,Consumer,South
2,CA-2016-138688,DV-13045,West,14.6200,6.8714,Corporate,West
3,US-2015-108966,SO-20335,South,957.5775,-383.0310,Consumer,South
4,US-2015-108966,SO-20335,South,22.3680,2.5164,Consumer,South


Right join shape: (113, 7)


In [11]:
outer_join = pd.merge(
    sales_df,
    customer_df,
    on="customer_id",
    how="outer",
    suffixes=("_sales", "_customer"),
    indicator=True
)

display(outer_join.head())

print("Outer join shape:", outer_join.shape)

print("\nJoin source:")
print(outer_join["_merge"].value_counts())

,order_id,customer_id,region_sales,sales,profit,segment,region_customer,_merge
0,CA-2017-114412,AA-10480,South,15.552,5.4432,Consumer,South,both
1,NaN,AD-10180,NaN,NaN,NaN,Home Office,Central,right_only
2,CA-2014-167164,AG-10270,West,55.500,9.9900,Consumer,West,both
3,CA-2014-115812,BH-11710,West,48.860,14.1694,Consumer,West,both
4,CA-2014-115812,BH-11710,West,7.280,1.9656,Consumer,West,both


Outer join shape: (113, 8)

Join source:
_merge
both          100
right_only     13
left_only       0
Name: count, dtype: int64


In [12]:
sales_pivot = pd.pivot_table(
    df,
    values="sales",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

display(sales_pivot)

category,Furniture,Office Supplies,Technology,Total
region,,,,
Central,163797.1638,167026.415,170416.312,5.012399e+05
East,208291.2040,205516.055,264973.981,6.787812e+05
South,117298.6840,125651.313,148771.908,3.917219e+05
West,252612.7435,220853.249,251991.832,7.254578e+05
Total,741999.7953,719047.032,836154.033,2.297201e+06


In [13]:
profit_pivot = pd.pivot_table(
    df,
    values="profit",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

display(profit_pivot)

category,Furniture,Office Supplies,Technology,All
region,,,,
Central,-2871.0494,8879.9799,33697.4320,39706.3625
East,3046.1658,41014.5791,47462.0351,91522.7800
South,6771.2061,19986.3928,19991.8314,46749.4303
West,11504.9503,52609.8490,44303.6496,108418.4489
All,18451.2728,122490.8008,145454.9481,286397.0217


In [14]:
# Create a wide-format sales summary
wide_sales = pd.pivot_table(
    df,
    values="sales",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

wide_sales = wide_sales.reset_index()

print("Wide-format data:")
display(wide_sales)

# Convert wide data into long format
long_sales = pd.melt(
    wide_sales,
    id_vars=["region"],
    var_name="category",
    value_name="total_sales"
)

print("Long-format data:")
display(long_sales)

print("Wide shape:", wide_sales.shape)
print("Long shape:", long_sales.shape)

Wide-format data:


category,region,Furniture,Office Supplies,Technology
0,Central,163797.1638,167026.415,170416.312
1,East,208291.2040,205516.055,264973.981
2,South,117298.6840,125651.313,148771.908
3,West,252612.7435,220853.249,251991.832


Long-format data:


,region,category,total_sales
0,Central,Furniture,163797.1638
1,East,Furniture,208291.2040
2,South,Furniture,117298.6840
3,West,Furniture,252612.7435
4,Central,Office Supplies,167026.4150
5,East,Office Supplies,205516.0550
6,South,Office Supplies,125651.3130
7,West,Office Supplies,220853.2490
8,Central,Technology,170416.3120
9,East,Technology,264973.9810


Wide shape: (4, 4)
Long shape: (12, 3)
